# Client and Answer Object Demo

The `PxFQuery` client exposes a scanpy-style workflow and a reusable answer object. Query data can be saved and reloaded for later presentation-layer outputs.

In [1]:
from pathlib import Path
import os
import sys
import warnings
from IPython.display import display

warnings.filterwarnings("ignore", message="IProgress not found.*")

repo_root = Path.cwd()
if not (repo_root / "src" / "pxfquery").exists() and (repo_root.parent / "src" / "pxfquery").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

# Optional: load local environment variables for the LLM provider.
for env_file in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.home() / ".env"]:
    if env_file.exists():
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip())

from pxfquery import PxFQuery

pxf = PxFQuery()
print("PxFquery", pxf.version)
resource_status = pxf.resources.status()
resource_info = resource_status.to_dict() if hasattr(resource_status, "to_dict") else dict(resource_status)
print("Resource status:")
print({
    "available": resource_info.get("available"),
    "source": resource_info.get("source"),
    "version": resource_info.get("version"),
    "available_file_count": len(resource_info.get("available_files") or {}),
})


PxFquery 0.5.12.dev0
Resource status:
{'available': True, 'source': 'manifest', 'version': 'v20260628', 'available_file_count': 18}


In [2]:
question = "In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout?"
print("Question:")
print(question)

qdata = pxf.tl.parse(question, top_n=8)
pxf.tl.answer(qdata)
answer = pxf.get.answer(qdata)

print("\nSummary:")
print(answer.summary)
print("\nObject fields:")
print(list(answer.to_dict().keys()))

Question:
In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout?


[Parsing] start


[Parsing] done | mode=forward; context=A549 lung cancer cells; perturbation=EGFR; time=1.57s
[Matching] start


[Matching] done | matches=6; time=5.36s
[Matrix] start


[Matrix] done | profiles=6; skipped=0; time=0.73s
[Evidence] start


[Evidence] done | status=ready; time=4.65s



Summary:
In A549 lung cancer cells, EGFR CRISPR knockout is associated with increased interferon responses (alpha and gamma), EMT-I, and adipogenesis, and decreased cell cycle (G2/M), MYC targets, E2F targets, and MYC program.

Object fields:
['question', 'interpreted_question', 'headline', 'summary', 'summary_source', 'biological_results', 'evidence', 'limitations', 'tables', 'figures', 'html', 'mcp', 'rendering_contract', 'structured_result', 'engineering']


In [3]:
print("Tables:")
print(list(answer.tables.keys()))
print("\nFigures:")
print([fig.get("kind") for fig in answer.figures])
print("\nTop biological results:")
display(answer.biological_results[:5])

Tables:
['ranked_results', 'primary_route_ranked_results', 'route_summary', 'route_target_functions', 'route_function_results', 'matrix_context', 'claim_rules']

Figures:
['evidence_match_map', 'function_match_heatmap', 'function_consensus_bar']

Top biological results:


[{'rank': 1,
  'label': 'HALLMARK_MYC_TARGETS_V2',
  'score': 5.359375,
  'direction': 'activated',
  'kind': 'activated_function',
  'source': None},
 {'rank': 2,
  'label': 'MP11 Translation initiation',
  'score': 5.35546875,
  'direction': 'activated',
  'kind': 'activated_function',
  'source': None},
 {'rank': 3,
  'label': 'MP8 Proteasomal degradation',
  'score': 5.33984375,
  'direction': 'activated',
  'kind': 'activated_function',
  'source': None},
 {'rank': 4,
  'label': 'MP25 Astrocytes',
  'score': 5.0,
  'direction': 'activated',
  'kind': 'activated_function',
  'source': None},
 {'rank': 5,
  'label': 'MP20 MYC',
  'score': 5.0,
  'direction': 'activated',
  'kind': 'activated_function',
  'source': None}]

In [4]:
path = Path("egfr_crispr_query.pkl")
pxf.tl.save(qdata, path)
restored = pxf.tl.load(path)
restored_answer = pxf.get.answer(restored)
print("Saved query object:", path)
print("Restored summary preview:")
print(restored_answer.summary[:500])

Saved query object: egfr_crispr_query.pkl
Restored summary preview:
In A549 lung cancer cells, EGFR CRISPR knockout is associated with increased interferon responses (alpha and gamma), EMT-I, and adipogenesis, and decreased cell cycle (G2/M), MYC targets, E2F targets, and MYC program.


In [5]:
quick_answer = pxf.ask("For melanoma models treated with BRAF inhibitors, which functional programs change?")
print("Convenience API answer type:")
print(type(quick_answer))
print("\nSummary preview:")
print(quick_answer.summary[:500])

[Parsing] start


[Parsing] done | mode=forward; context=melanoma models; perturbation=BRAF inhibitors; time=1.50s
[Matching] start


[Matching] done | matches=6; time=2.53s
[Matrix] start


[Matrix] done | profiles=6; skipped=0; time=1.01s
[Evidence] start


Convenience API answer type:
<class 'pxfquery.l5_presentation.model.PxFQueryAnswer'>

Summary preview:
In melanoma models treated with BRAF inhibitors, the most consistent changes across cell lines are increased interferon response, myogenesis, and oligodendrocyte-related programs, and decreased cell cycle, MYC, and mTORC1 signaling programs. However, the specific programs vary by cell line, indicating a mixed response. For example, in A375 cells, interferon alpha response and myogenesis are activated while cell cycle G2/M and MYC are suppressed; in SH4 cells, KRAS signaling down is activated and


[Evidence] done | status=ready; time=8.44s
